<a href="https://colab.research.google.com/github/arups330/ElitLab_MED_VQA/blob/main/Abdomen_open__WITHOUT_CoT_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [Colab] Open-question fine-tuning **WITHOUT CoT** — all models, one after another, auto-push to Hugging Face

**Before running**
* Runtime → Change runtime type → **GPU** (T4).
* Upload your dataset **.zip** to `/content` via the Files panel (left sidebar). Cell 5 extracts it and finds `open_with_CoT.csv` automatically.
* 🔑 **Secrets** (left sidebar) → add `HF_TOKEN` = your Hugging Face **Write** token → enable *Notebook access*.
* Accept the license at https://huggingface.co/google/medgemma-4b-it.

This is the open-question counterpart to the closed-question notebook. The only real
difference is the data/target: open questions have **free-text answers (1–4 words)**,
not a fixed Yes/No set, so there's nothing to filter against — rows are kept as long as
they have a non-empty answer. Everything else (per-model steps, auto-skip already-pushed
models, GPU cleanup between models) is identical.

Cells 7–14 are each one step of the original notebook (load → LoRA → before check → trainer → train → after check → save → push → free GPU). Cell 15 runs them for every model in `REPO_IDS`, pushing each to the Hub before the next. Models already on the Hub are skipped — if the summary shows an OOM, **Runtime → Restart session** and run again.


In [ ]:
!pip install -q "unsloth==2026.9.2"


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # before torch is imported
os.environ["UNSLOTH_RETURN_LOGITS"]   = "1"   # bypass Unsloth fused CE loss (Llama torch.compile crash)
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"   # disable torch.compile patches (same maths, a bit slower)

from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch, gc, shutil

REPO_IDS = {
    "MedGemma-4B": "google/medgemma-4b-it",
    "Qwen2.5-VL-7B": "Qwen/Qwen2.5-VL-7B-Instruct",
    "Llama-11B": "unsloth/Llama-3.2-11B-Vision-Instruct",
    "Qwen3-VL-8B": "Qwen/Qwen3-VL-8B-Instruct",
    "Lingshu-7B": "lingshu-medical-mllm/Lingshu-7B",
    # "Gemma4-E4B": "google/gemma-4-E4B-it",   # not trainable on T4 -> needs Colab L4/A100
}

HF_USERNAME = "Arup330"
DATASET_TAG = "Abdomen"


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
# HF token from Colab Secrets (🔑 icon in left sidebar -> add HF_TOKEN, enable notebook access)
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN


In [ ]:
# ---------------------------------------------------------------------------
# 3. Upload zip -> extract -> find open_with_CoT.csv -> keep rows with a valid answer
# ---------------------------------------------------------------------------
import os
import glob
import zipfile
import pandas as pd
from PIL import Image

EXTRACT_DIR = "/content/data"

# Use a zip already uploaded to /content (Files panel), otherwise open the upload dialog
zips = glob.glob("/content/*.zip")
if not zips:
    from google.colab import files
    uploaded = files.upload()
    zips = [f"/content/{n}" for n in uploaded.keys()]
ZIP_PATH = zips[0]
print("Using zip:", ZIP_PATH)

os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(EXTRACT_DIR)
print("Extracted to", EXTRACT_DIR)

# Locate open_with_CoT.csv anywhere inside the extracted folder (case-insensitive)
csv_paths = [p for p in glob.glob(os.path.join(EXTRACT_DIR, "**", "*.csv"), recursive=True)
             if os.path.basename(p).lower() == "open_with_cot.csv"]
assert csv_paths, "open_with_CoT.csv not found inside the zip"
print(f"Found {len(csv_paths)} CSVs:")
for p in csv_paths:
    print(" ", p)

frames = []
for csv_path in csv_paths:
    split_dir = os.path.dirname(csv_path)
    df = pd.read_csv(csv_path)
    df["split_dir"] = split_dir
    frames.append(df)

open_df = pd.concat(frames, ignore_index=True)

# Open-ended answers are free text (not Yes/No), so there's no fixed value set to
# filter against -- just drop rows with a missing/empty answer.
before = len(open_df)
open_df["answer"] = open_df["answer"].astype(str).str.strip()
open_df = open_df[(open_df["answer"] != "") & (open_df["answer"].str.lower() != "nan")].copy()
open_df = open_df.reset_index(drop=True)

# Sanity check on answer length, since the prompt asks the model for a 1-4 word
# answer: flag (but don't silently drop) anything longer, so you can decide.
word_counts = open_df["answer"].str.split().str.len()
long_mask = word_counts > 4
print(f"Kept {len(open_df)} / {before} rows with a non-empty answer "
      f"({int(long_mask.sum())} have answers longer than 4 words -- inspect these "
      f"if you want training targets to strictly match the 1-4 word output format).")
if long_mask.any():
    print(open_df.loc[long_mask, "answer"].value_counts().head(10))

IMG_COL = "image_file" if "image_file" in open_df.columns else "img_name"

# Index every image inside the zip by file name (images may be in any sub-folder)
IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".gif", ".webp")
IMG_INDEX = {os.path.basename(p): p
             for p in glob.glob(os.path.join(EXTRACT_DIR, "**", "*"), recursive=True)
             if p.lower().endswith(IMG_EXTS)}
print(f"Indexed {len(IMG_INDEX)} image files")

def resolve_image_path(split_dir: str, img_name: str) -> str:
    flat_name = os.path.basename(str(img_name))
    candidates = [
        os.path.join(split_dir, str(img_name)),
        os.path.join(split_dir, flat_name),
        os.path.join(split_dir, str(img_name).replace("/", "_")),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    if flat_name in IMG_INDEX:
        return IMG_INDEX[flat_name]
    raise FileNotFoundError(f"Could not find image for {img_name!r} in {split_dir}")

print("First image:", resolve_image_path(open_df.iloc[0]["split_dir"], open_df.iloc[0][IMG_COL]))


Using zip: /content/Abdomen_Train_with_CoT.zip
Extracted to /content/data
Found 1 CSVs:
  /content/data/Abdomen_Train_with_CoT/CT/train/open_with_CoT.csv
Kept 150 / 150 rows with a non-empty answer (0 have answers longer than 4 words -- inspect these if you want training targets to strictly match the 1-4 word output format).
Indexed 92 image files
First image: /content/data/Abdomen_Train_with_CoT/CT/train/xmlab104_source.jpg


In [ ]:
# ---------------------------------------------------------------------------
# 4. convert_to_conversation -- instruction = your OPEN-question prompt, and the
#    assistant target is the ground-truth free-text answer (1-4 words), used as-is.
# ---------------------------------------------------------------------------
def systemPrompt(question: str) -> str:
    return f"""Context:
You are a board-certified radiologist and Medical Visual Question Answering (MedVQA) expert with experience interpreting X-ray, CT, MRI, Ultrasound, and other medical images.

Objective:
Answer the user's question directly based strictly on the visual evidence in the medical image.

Inputs:
Question: {question}

Instructions:
1. Examine the medical image carefully.
2. Identify all relevant anatomical structures and pathological findings.
3. Formulate the direct clinical answer to the question using only visual evidence.
4. Never fabricate findings or make unsupported assumptions.

Output Requirements:
- Return ONLY the concise final answer in 1 to 4 words.
- Do not provide explanations, full sentences, or punctuation.
"""

def convert_to_conversation(sample):
    instruction = systemPrompt(sample["question"])
    image_path = resolve_image_path(sample["split_dir"], sample[IMG_COL])
    image = Image.open(image_path).convert("RGB")

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": instruction},
                {"type": "image", "image": image},
            ],
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": sample["answer"]}],
        },
    ]
    return {"messages": conversation}
pass

print("Converting rows to Unsloth chat format...")
converted_dataset = [convert_to_conversation(row) for _, row in open_df.iterrows()]
print(f"Done. {len(converted_dataset)} examples ready.")
print("Example:", converted_dataset[0]["messages"])


Converting rows to Unsloth chat format...
Done. 150 examples ready.
Example: [{'role': 'user', 'content': [{'type': 'text', 'text': "Context:\nYou are a board-certified radiologist and Medical Visual Question Answering (MedVQA) expert with experience interpreting X-ray, CT, MRI, Ultrasound, and other medical images.\n\nObjective:\nAnswer the user's question directly based strictly on the visual evidence in the medical image.\n\nInputs:\nQuestion: What modality is used to take this image?\n\nInstructions:\n1. Examine the medical image carefully.\n2. Identify all relevant anatomical structures and pathological findings.\n3. Formulate the direct clinical answer to the question using only visual evidence.\n4. Never fabricate findings or make unsupported assumptions.\n\nOutput Requirements:\n- Return ONLY the concise final answer in 1 to 4 words.\n- Do not provide explanations, full sentences, or punctuation.\n"}, {'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=512x512 at 0x

## Per-model steps (each cell = one step of the original notebook)


In [ ]:
# ---------------------------------------------------------------------------
# 1. Load model
# ---------------------------------------------------------------------------
def load_model(repo_id):
    extra = {}
    if "Qwen3-VL" in repo_id:
        extra["device_map"] = {"": 0}   # Qwen3-VL vision encoder breaks when split across GPUs
    model, tokenizer = FastVisionModel.from_pretrained(
        repo_id,
        load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
        use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
        token = HF_TOKEN,
        **extra,
    )
    return model, tokenizer


In [ ]:
# ---------------------------------------------------------------------------
# 2. Attach LoRA adapters
# ---------------------------------------------------------------------------
def attach_lora(model, model_name):
    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers=(model_name != "MedGemma-4B"),   # T4 has no bf16: keep MedGemma's vision tower frozen
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=16,
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        random_state=3407,
        use_rslora=False,
        loftq_config=None,
    )
    if model_name == "MedGemma-4B":
        # T4 trains Gemma3 in float32; upcast any bf16 weights left in the vision tower so layer_norm dtypes match
        for p in model.parameters():
            if p.dtype == torch.bfloat16:
                p.data = p.data.float()
        for b in model.buffers():
            if b.dtype == torch.bfloat16:
                b.data = b.data.float()
    return model


In [ ]:
# ---------------------------------------------------------------------------
# 5. Quick check BEFORE fine-tuning (same as the Unsloth notebook does --
#    run one inference with the base model to see what it currently outputs)
# ---------------------------------------------------------------------------
def check_before(model, tokenizer):
    FastVisionModel.for_inference(model)
    sample = open_df.iloc[0]
    test_instruction = systemPrompt(sample["question"])
    test_image = Image.open(resolve_image_path(sample["split_dir"], sample[IMG_COL])).convert("RGB")

    messages = [{"role": "user", "content": [
        {"type": "image"}, {"type": "text", "text": test_instruction}
    ]}]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(test_image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
    output_ids = model.generate(**inputs, max_new_tokens=32, use_cache=True, temperature=1.5, min_p=0.1)
    print("\nBEFORE fine-tuning, model output:")
    print(tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
    print("Ground truth:", sample["answer"])
    return inputs, sample


In [ ]:
# ---------------------------------------------------------------------------
# 6. Train with SFTTrainer + UnslothVisionDataCollator
# ---------------------------------------------------------------------------
from trl import SFTTrainer, SFTConfig
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator

def build_trainer(model, tokenizer):
    FastVisionModel.for_training(model)

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        data_collator=UnslothVisionDataCollator(model, tokenizer),
        train_dataset=converted_dataset,
        args=SFTConfig(
            per_device_train_batch_size=2,   # kept identical to the closed-question notebook for
            gradient_accumulation_steps=4,   # consistency across all five backbones on a single T4;
            warmup_steps=5,                  # answers here are short (1-4 words) too, so feel free
            max_steps=30,                  # quick test run -- comment out for full training
            num_train_epochs=1,              # to raise the batch size if you have more VRAM.
            learning_rate=2e-4,
            fp16=not is_bf16_supported(),
            bf16=is_bf16_supported(),
            logging_steps=1,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            seed=3407,
            output_dir="outputs",
            report_to="none",

            remove_unused_columns=False,
            dataset_text_field="",
            dataset_kwargs={"skip_prepare_dataset": True},
            dataset_num_proc=4,
            max_length=4096,   # generous headroom for prompt + image tokens; the target answer itself is short (1-4 words)
        ),
    )
    return trainer


In [ ]:
# ---------------------------------------------------------------------------
# 7. Check AFTER fine-tuning -- same sample as before, compare outputs
# ---------------------------------------------------------------------------
def check_after(model, tokenizer, inputs, sample):
    FastVisionModel.for_inference(model)
    output_ids = model.generate(**inputs, max_new_tokens=32, use_cache=True, temperature=0.3, min_p=0.1)
    print("\nAFTER fine-tuning, model output:")
    print(tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
    print("\n(Ground truth answer it was trained on:)")
    print(sample["answer"])


In [ ]:
# ---------------------------------------------------------------------------
# 8. Save the LoRA adapter locally
# ---------------------------------------------------------------------------
def save_local(model, tokenizer, local_dir):
    model.save_pretrained(local_dir)
    tokenizer.save_pretrained(local_dir)
    print(f"\nSaved locally to ./{local_dir}")


In [ ]:
# ---------------------------------------------------------------------------
# 9. Push to Hugging Face Hub
# ---------------------------------------------------------------------------
def push_to_hf(model, tokenizer, hub_repo):
    model.push_to_hub(hub_repo, token=HF_TOKEN)
    tokenizer.push_to_hub(hub_repo, token=HF_TOKEN)
    print(f"Pushed -> https://huggingface.co/{hub_repo}")


In [ ]:
# ---------------------------------------------------------------------------
# 10. Free GPU memory + HF download cache before the next model
# ---------------------------------------------------------------------------
def free_gpu():
    for v in ["trainer", "model", "tokenizer", "inputs"]:
        if v in globals(): del globals()[v]
    gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub"), ignore_errors=True)   # keeps Colab disk free
    print("GPU free after cleanup:", [f"{torch.cuda.mem_get_info(i)[0]/1e9:.1f} GB" for i in range(torch.cuda.device_count())])


## Run all models (one after another; each pushed to the Hub before the next starts)


In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
results = {}

for model_name, repo_id in REPO_IDS.items():
    HUB_REPO  = f"{HF_USERNAME}/{DATASET_TAG}_open_noCoT_{model_name}_lora"
    LOCAL_DIR = f"{DATASET_TAG}_open_noCoT_{model_name}_lora"

    if api.repo_exists(HUB_REPO):          # already trained + pushed -> skip (safe re-run after a session ends)
        print(f"\n[SKIP] {model_name} already on Hub: https://huggingface.co/{HUB_REPO}")
        results[model_name] = "skipped"
        continue

    print("\n" + "="*70)
    print(f"MODEL: {model_name}  ({repo_id})  ->  {HUB_REPO}")
    print("="*70)

    try:
        model, tokenizer = load_model(repo_id)                 # 1
        model            = attach_lora(model, model_name)      # 2
        inputs, sample   = check_before(model, tokenizer)      # 5
        trainer          = build_trainer(model, tokenizer)     # 6
        trainer_stats    = trainer.train()
        check_after(model, tokenizer, inputs, sample)          # 7
        save_local(model, tokenizer, LOCAL_DIR)                # 8
        push_to_hf(model, tokenizer, HUB_REPO)                 # 9
        results[model_name] = "OK"
    except Exception as e:
        import traceback; traceback.print_exc()
        results[model_name] = f"FAILED: {e}"

    free_gpu()                                                 # 10

print("\n================ SUMMARY ================")
for k, v in results.items():
    print(f"{k:15s} {v}")



MODEL: MedGemma-4B  (google/medgemma-4b-it)  ->  Arup330/Abdomen_open_noCoT_MedGemma-4B_lora
==((====))==  Unsloth 2026.9.2: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]


BEFORE fine-tuning, model output:
CT

Ground truth: CT


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 150 | Num Epochs = 2 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,802,496 of 4,329,881,968 (0.69% trained)
Unsloth: Not an error, but Gemma3ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


...

Unsloth: Offloading embeddings to RAM to save 1.02 GB.


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None}.



BEFORE fine-tuning, model output:
CT scan.
Ground truth: CT
Unsloth: Model does not have a default image size - using 512


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 150 | Num Epochs = 2 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 51,521,536 of 8,343,688,192 (0.62% trained)
Traceback (most recent call last):
  File "/tmp/ipykernel_1469/4026324239.py", line 23, in <cell line: 0>
    trainer_stats    = trainer.train()
  File "/usr/local/lib/python3.13/dist-packages/unsloth/trainer.py", line 1049, in _train_with_reset
    return _orig_train(*train_args, **train_kwargs)
  File "/usr/local/lib/python3.13/dist-packages/transformers/trainer.py", line 1424, in train
    return inner_training_loop(
        args=args,
    ...<2 lines>...
        ignore_keys_for_eval=ignore_keys_for_eval,
    )
  File "<string>", line 81, in _fast_inner_training_loop
  File "/usr/local/lib/python3.13/dist-packages/transform

GPU free after cleanup: ['5.8 GB']

================ SUMMARY ================
MedGemma-4B     OK
Qwen2.5-VL-7B   OK
Llama-11B       FAILED: CUDA out of memory. Tried to allocate 252.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 231.81 MiB is free. Including non-PyTorch memory, this process has 14.33 GiB memory in use. Of the allocated memory 14.00 GiB is allocated by PyTorch, and 176.60 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)
Qwen3-VL-8B     OK
Lingshu-7B      FAILED: CUDA out of memory. Tried to allocate 584.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 531.81 MiB is free. Including non-PyTorch memory, this process has 14.04 GiB memory in use. Of the allocated memory 13.66 GiB is alloc